# Stress Distribution Calculation

This notebook demonstrates the packaged stress workflow. Stress calculation requires `open3d` for normal estimation.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

src_path = PROJECT_ROOT / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

DATA_DIR = PROJECT_ROOT / "data" / "raw"
OUT_DIR = PROJECT_ROOT / "outputs" / "notebooks"
print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")

## Run the workflow

In [ ]:
from pointcloud_etfe_postprocessing.config import TriangulationConfig
from pointcloud_etfe_postprocessing.workflows import run_stress_workflow

result = run_stress_workflow(
    DATA_DIR / "zxt_14000Pa_failure.xlsx",
    OUT_DIR / "stress",
    triangulation=TriangulationConfig(method="structured", boundary_scale=1.8),
    plot=True,
)
result.paths

## Inspect point-level stress

In [ ]:
point_stress_csv = OUT_DIR / "stress" / "zxt_14000Pa_failure_point_stress.csv"
point_stress = pd.read_csv(point_stress_csv)
print(f"Rows: {len(point_stress)}")
point_stress[["Sxx", "Syy", "principal_stress", "mises_stress"]].agg(["min", "mean", "max"]).round(3)

In [ ]:
point_stress.head()

## Inspect link-level stress

In [ ]:
links_x = pd.read_csv(OUT_DIR / "stress" / "zxt_14000Pa_failure_links_x_stress.csv")
links_y = pd.read_csv(OUT_DIR / "stress" / "zxt_14000Pa_failure_links_y_stress.csv")
summary = pd.DataFrame(
    {
        "direction": ["x", "y"],
        "links": [len(links_x), len(links_y)],
        "mean_stress": [links_x["stress"].mean(), links_y["stress"].mean()],
        "max_stress": [links_x["stress"].max(), links_y["stress"].max()],
    }
)
summary.round(3)

## Preview the von Mises stress field

In [ ]:
stress_png = OUT_DIR / "stress" / "zxt_14000Pa_failure_mises_stress.png"
display(Image(filename=stress_png))